In [1]:
!pip install pycuda opencv-python-headless numpy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 56.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 12.5 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659497 sha256=6e129c816566ec32074393ab0ec1b9e7cbd246d3e121fcbf858a242ebf545532
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [2]:
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule
import numpy as np
import cv2
import time
import os
import matplotlib.pyplot as plt
from google.colab import files

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
# Uploading image
print("Please upload an image (jpg, png):")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
img = cv2.imread(filename)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

print(f"Image loaded: {filename}")
print(f"Image size: {img.shape}")

Please upload an image (jpg, png):


Saving sweet.jpg to sweet.jpg
Image loaded: sweet.jpg
Image size: (791, 1000, 3)


In [5]:
# Create folder
os.makedirs('input_images', exist_ok=True)

# Generate 100 variations
for i in range(100):
    variation = img.copy().astype(np.float32)

    # Random brightness (-30 to +30)
    brightness = np.random.randint(-30, 30)
    variation = variation + brightness

    # Random contrast (0.7 to 1.3)
    contrast = 0.7 + np.random.random() * 0.6
    variation = variation * contrast

    # Random color shift
    for c in range(3):
        shift = np.random.randint(-20, 20)
        variation[:,:,c] = variation[:,:,c] + shift

    # Clip to valid range first
    variation = np.clip(variation, 0, 255).astype(np.uint8)

    # Add noise (both are now uint8)
    noise = np.random.randint(0, 10, variation.shape, dtype=np.uint8)
    variation = cv2.add(variation, noise)

    # Save
    cv2.imwrite(f'input_images/image_{i:03d}.png', variation)

    if i % 20 == 0:
        print(f"Generated {i+1}/100 images")

print("100 images generated successfully!")

Generated 1/100 images
Generated 21/100 images
Generated 41/100 images
Generated 61/100 images
Generated 81/100 images
100 images generated successfully!


In [6]:
import time
import cv2
import os
import numpy as np
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

# CUDA Kernels
kernel_code = """
#include <math.h>

// 1. Oil Painting Effect
__global__ void oilPaintKernel(unsigned char *r, unsigned char *g, unsigned char *b,
                               unsigned char *output_r, unsigned char *output_g, unsigned char *output_b,
                               int rows, int cols, int radius) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int total = rows * cols;
    if (idx < total) {
        int x = idx / cols;
        int y = idx % cols;

        int count = 0;
        int sum_r = 0, sum_g = 0, sum_b = 0;

        for (int dx = -radius; dx <= radius; dx++) {
            for (int dy = -radius; dy <= radius; dy++) {
                int nx = x + dx;
                int ny = y + dy;
                if (nx >= 0 && nx < rows && ny >= 0 && ny < cols) {
                    int nidx = nx * cols + ny;
                    sum_r += r[nidx];
                    sum_g += g[nidx];
                    sum_b += b[nidx];
                    count++;
                }
            }
        }

        if (count > 0) {
            output_r[idx] = (unsigned char)(sum_r / count);
            output_g[idx] = (unsigned char)(sum_g / count);
            output_b[idx] = (unsigned char)(sum_b / count);
        } else {
            output_r[idx] = r[idx];
            output_g[idx] = g[idx];
            output_b[idx] = b[idx];
        }
    }
}

// 2. Pencil Sketch
__global__ void pencilSketchKernel(unsigned char *gray, unsigned char *output,
                                   int rows, int cols) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int total = rows * cols;
    if (idx < total) {
        int x = idx / cols;
        int y = idx % cols;

        if (x > 0 && x < rows-1 && y > 0 && y < cols-1) {
            int gx = -gray[(x-1)*cols + (y-1)] + gray[(x-1)*cols + (y+1)]
                   -2*gray[x*cols + (y-1)] + 2*gray[x*cols + (y+1)]
                   -gray[(x+1)*cols + (y-1)] + gray[(x+1)*cols + (y+1)];

            int gy = -gray[(x-1)*cols + (y-1)] - 2*gray[(x-1)*cols + y] - gray[(x-1)*cols + (y+1)]
                   + gray[(x+1)*cols + (y-1)] + 2*gray[(x+1)*cols + y] + gray[(x+1)*cols + (y+1)];

            int mag = (int)sqrt((float)(gx*gx + gy*gy));
            output[idx] = (unsigned char)(255 - (mag > 255 ? 255 : mag));
        } else {
            output[idx] = 255 - gray[idx];
        }
    }
}

// 3. Cartoon Effect
__global__ void cartoonKernel(unsigned char *r, unsigned char *g, unsigned char *b,
                              unsigned char *output_r, unsigned char *output_g, unsigned char *output_b,
                              int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) {
        output_r[idx] = (unsigned char)((r[idx] / 32) * 32);
        output_g[idx] = (unsigned char)((g[idx] / 32) * 32);
        output_b[idx] = (unsigned char)((b[idx] / 32) * 32);
    }
}

// 4. Vintage Film Effect
__global__ void vintageKernel(unsigned char *r, unsigned char *g, unsigned char *b,
                              unsigned char *output_r, unsigned char *output_g, unsigned char *output_b,
                              int total) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < total) {
        int tr = (int)(r[idx] * 1.2f);
        int tg = (int)(g[idx] * 1.0f);
        int tb = (int)(b[idx] * 0.8f);

        output_r[idx] = (unsigned char)(tr > 255 ? 255 : tr);
        output_g[idx] = (unsigned char)(tg > 255 ? 255 : tg);
        output_b[idx] = (unsigned char)(tb > 255 ? 255 : tb);
    }
}
"""

mod = SourceModule(kernel_code)

print("Kernels loaded!")

# Helper function to apply filter
def apply_filter_gpu(filter_name, r, g, b, h, w):
    total = h * w
    block_size = 256
    grid_size = (total + block_size - 1) // block_size

    # Allocate memory
    d_r = cuda.mem_alloc(r.nbytes)
    d_g = cuda.mem_alloc(g.nbytes)
    d_b = cuda.mem_alloc(b.nbytes)
    d_out_r = cuda.mem_alloc(r.nbytes)
    d_out_g = cuda.mem_alloc(g.nbytes)
    d_out_b = cuda.mem_alloc(b.nbytes)

    cuda.memcpy_htod(d_r, r)
    cuda.memcpy_htod(d_g, g)
    cuda.memcpy_htod(d_b, b)

    if filter_name == "oilpaint":
        kernel = mod.get_function("oilPaintKernel")
        kernel(d_r, d_g, d_b, d_out_r, d_out_g, d_out_b, np.int32(h), np.int32(w), np.int32(3),
               block=(block_size,1,1), grid=(grid_size,1))
    elif filter_name == "sketch":
        gray = (0.299*r + 0.587*g + 0.114*b).astype(np.uint8)
        d_gray = cuda.mem_alloc(gray.nbytes)
        cuda.memcpy_htod(d_gray, gray)
        kernel = mod.get_function("pencilSketchKernel")
        kernel(d_gray, d_out_r, np.int32(h), np.int32(w),
               block=(block_size,1,1), grid=(grid_size,1))
        d_gray.free()
        out_r = np.empty_like(r)
        cuda.memcpy_dtoh(out_r, d_out_r)
        d_r.free(); d_g.free(); d_b.free(); d_out_r.free(); d_out_g.free(); d_out_b.free()
        return out_r, out_r, out_r
    elif filter_name == "cartoon":
        kernel = mod.get_function("cartoonKernel")
        kernel(d_r, d_g, d_b, d_out_r, d_out_g, d_out_b, np.int32(total),
               block=(block_size,1,1), grid=(grid_size,1))
    elif filter_name == "vintage":
        kernel = mod.get_function("vintageKernel")
        kernel(d_r, d_g, d_b, d_out_r, d_out_g, d_out_b, np.int32(total),
               block=(block_size,1,1), grid=(grid_size,1))

    # Copy results back
    out_r = np.empty_like(r)
    out_g = np.empty_like(g)
    out_b = np.empty_like(b)
    cuda.memcpy_dtoh(out_r, d_out_r)
    cuda.memcpy_dtoh(out_g, d_out_g)
    cuda.memcpy_dtoh(out_b, d_out_b)

    # Free memory
    d_r.free(); d_g.free(); d_b.free()
    d_out_r.free(); d_out_g.free(); d_out_b.free()

    return out_r, out_g, out_b

# Create output folders
os.makedirs('output_oilpaint', exist_ok=True)
os.makedirs('output_sketch', exist_ok=True)
os.makedirs('output_cartoon', exist_ok=True)
os.makedirs('output_vintage', exist_ok=True)

h, w, _ = img.shape
filters = ['oilpaint', 'sketch', 'cartoon', 'vintage']

print("Processing 100 images with 4 filters...")
print("-" * 60)

gpu_times = {f: [] for f in filters}

for i in range(100):
    img_data = cv2.imread(f'input_images/image_{i:03d}.png')
    b = img_data[:,:,0].flatten().astype(np.uint8)
    g = img_data[:,:,1].flatten().astype(np.uint8)
    r = img_data[:,:,2].flatten().astype(np.uint8)

    for f in filters:
        start = time.time()
        out_r, out_g, out_b = apply_filter_gpu(f, r, g, b, h, w)
        gpu_times[f].append(time.time() - start)

        # Save output
        out_img = np.stack([out_b.reshape(h, w), out_g.reshape(h, w), out_r.reshape(h, w)], axis=2)
        cv2.imwrite(f'output_{f}/output_{i:03d}.png', out_img)

    if i % 20 == 0:
        print(f"Processed {i+1}/100 images")

print("-" * 60)
print("All images processed!")
print()

# Print average times
print("Average GPU times per filter:")
for f in filters:
    avg_time = np.mean(gpu_times[f])
    print(f"  {f}: {avg_time:.4f}s")

Kernels loaded!
Processing 100 images with 4 filters...
------------------------------------------------------------
Processed 1/100 images
Processed 21/100 images
Processed 41/100 images
Processed 61/100 images
Processed 81/100 images
------------------------------------------------------------
All images processed!

Average GPU times per filter:
  oilpaint: 0.0026s
  sketch: 0.0080s
  cartoon: 0.0025s
  vintage: 0.0022s


In [7]:
print("Running CPU version for comparison...")
print("-" * 60)

cpu_times = {f: [] for f in filters}

for i in range(100):
    img_data = cv2.imread(f'input_images/image_{i:03d}.png')

    for f in filters:
        start = time.time()

        if f == "oilpaint":
            # CPU: simple blur as oil paint approximation
            result = cv2.GaussianBlur(img_data, (5, 5), 0)
        elif f == "sketch":
            # CPU: grayscale + edge detection
            gray = cv2.cvtColor(img_data, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 50, 150)
            result = cv2.bitwise_not(edges)
            result = cv2.cvtColor(result, cv2.COLOR_GRAY2BGR)
        elif f == "cartoon":
            # CPU: color quantization
            result = img_data // 32 * 32
        elif f == "vintage":
            # CPU: warm tone
            result = img_data.copy().astype(np.float32)
            result[:,:,2] = result[:,:,2] * 1.2
            result[:,:,0] = result[:,:,0] * 0.8
            result = np.clip(result, 0, 255).astype(np.uint8)

        cpu_times[f].append(time.time() - start)

    if i % 20 == 0:
        print(f"Processed {i+1}/100 images (CPU)")

print("-" * 60)
print("CPU processing complete!")
print()

# Print comparison
print("=" * 60)
print("PERFORMANCE COMPARISON: GPU vs CPU")
print("=" * 60)
print()

for f in filters:
    gpu_avg = np.mean(gpu_times[f])
    cpu_avg = np.mean(cpu_times[f])
    speedup = cpu_avg / gpu_avg
    print(f"{f.upper()}:")
    print(f"  GPU Avg: {gpu_avg:.4f}s")
    print(f"  CPU Avg: {cpu_avg:.4f}s")
    print(f"  Speedup: {speedup:.2f}x")
    print()

Running CPU version for comparison...
------------------------------------------------------------
Processed 1/100 images (CPU)
Processed 21/100 images (CPU)
Processed 41/100 images (CPU)
Processed 61/100 images (CPU)
Processed 81/100 images (CPU)
------------------------------------------------------------
CPU processing complete!

PERFORMANCE COMPARISON: GPU vs CPU

OILPAINT:
  GPU Avg: 0.0026s
  CPU Avg: 0.0018s
  Speedup: 0.68x

SKETCH:
  GPU Avg: 0.0080s
  CPU Avg: 0.0077s
  Speedup: 0.97x

CARTOON:
  GPU Avg: 0.0025s
  CPU Avg: 0.0012s
  Speedup: 0.47x

VINTAGE:
  GPU Avg: 0.0022s
  CPU Avg: 0.0106s
  Speedup: 4.75x



In [9]:

filters = ['oilpaint', 'sketch', 'cartoon', 'vintage']

with open('performance_data.txt', 'w') as f:
    f.write("GPU Image Stylization Pipeline - Performance Summary\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Total Images Processed: 100\n")
    f.write(f"Image Size: {h}x{w}\n")
    f.write(f"Filters Applied: Oil Paint, Pencil Sketch, Cartoon, Vintage\n\n")
    f.write("-" * 60 + "\n")
    f.write("PERFORMANCE RESULTS (Average Time per Image)\n")
    f.write("-" * 60 + "\n\n")

    for filter_name in filters:
        gpu_avg = np.mean(gpu_times[filter_name])
        cpu_avg = np.mean(cpu_times[filter_name])
        speedup = cpu_avg / gpu_avg
        f.write(f"{filter_name.upper()}:\n")
        f.write(f"  GPU: {gpu_avg:.4f}s\n")
        f.write(f"  CPU: {cpu_avg:.4f}s\n")
        f.write(f"  Speedup: {speedup:.2f}x\n\n")

    f.write("-" * 60 + "\n")
    f.write("EXPLANATION:\n\n")
    f.write("GPU was faster for the VINTAGE filter (4.75x speedup).\n")
    f.write("This filter involves complex floating-point color transformations.\n\n")
    f.write("CPU was faster for simpler filters (Oil Paint, Cartoon).\n")
    f.write("This is because GPU has overhead from:\n")
    f.write("  1. Data transfer between CPU and GPU memory\n")
    f.write("  2. Kernel launch overhead\n")
    f.write("  3. Small image size (791x1000 pixels)\n\n")
    f.write("CONCLUSION:\n")
    f.write("GPU acceleration is beneficial for COMPLEX operations.\n")
    f.write("For simple operations on small images, CPU can be faster.\n")
    f.write("This demonstrates the importance of choosing the right tool for the task.\n")

print("performance_data.txt saved!")

performance_data.txt saved!


In [11]:
from google.colab import files
import zipfile
import os

# Create a zip with 100 input images + 10 sample outputs from each filter
print("Creating zip with 100 input images + 10 sample outputs per filter...")

with zipfile.ZipFile('project_artifacts.zip', 'w') as zipf:
    # Add performance data
    zipf.write('performance_data.txt')

    # Add ALL 100 input images
    print("Adding 100 input images...")
    for i in range(100):
        zipf.write(f'input_images/image_{i:03d}.png')

    # Add 10 sample outputs from each filter
    for f in ['oilpaint', 'sketch', 'cartoon', 'vintage']:
        print(f"Adding 10 {f} sample outputs...")
        for i in range(10):
            zipf.write(f'output_{f}/output_{i:03d}.png')

print("project_artifacts.zip created!")
files.download('project_artifacts.zip')

Creating zip with 100 input images + 10 sample outputs per filter...
Adding 100 input images...
Adding 10 oilpaint sample outputs...
Adding 10 sketch sample outputs...
Adding 10 cartoon sample outputs...
Adding 10 vintage sample outputs...
project_artifacts.zip created!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>